In [1]:
DATA_ROOT = "/kaggle/input/140k-real-and-fake-faces/real_vs_fake/real-vs-fake"
!ls -1 $DATA_ROOT; ls -1 $DATA_ROOT/train; ls -1 $DATA_ROOT/valid; ls -1 $DATA_ROOT/test


test
train
valid
fake
real
fake
real
fake
real


In [2]:
# === Stage-2B inline model & dataset (SLIC-enabled, CMF + FFT + Superpixel tokens) ===
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np
from torchvision.models import resnet50, ResNet50_Weights
from torchvision import transforms
from torchvision.datasets import ImageFolder
from skimage.segmentation import slic

from sklearn.metrics import roc_auc_score, f1_score, accuracy_score, roc_curve
from scipy.optimize import brentq
from scipy.interpolate import interp1d

def calc_metrics(y_true, y_prob):
    y_pred = (y_prob > 0.5).astype(int)
    auc  = roc_auc_score(y_true, y_prob)
    f1   = f1_score(y_true, y_pred)
    acc  = accuracy_score(y_true, y_pred)
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    eer  = brentq(lambda x: 1. - x - interp1d(fpr, tpr)(x), 0., 1.)
    return auc, f1, eer, acc


# ---------------- Dataset with SLIC ----------------
class SuperpixelImageFolder(ImageFolder):
    """
    Returns (x_norm, seg_ids, y)
    - x_norm: normalized tensor for the network
    - seg_ids: HxW int map (superpixel ids) computed on de-normalized RGB for SLIC
    """
    def __init__(self, root, n_segments=50, compactness=10.0, image_size=224):
        tfm = transforms.Compose([
            transforms.Resize((image_size, image_size)),
            transforms.ToTensor(),
            transforms.Normalize(mean=(0.485,0.456,0.406), std=(0.229,0.224,0.225)),
        ])
        super().__init__(root, transform=tfm)
        self.n_segments = int(n_segments)
        self.compactness = float(compactness)
        self.image_size = int(image_size)
        self.mean = torch.tensor([0.485,0.456,0.406])[:,None,None]
        self.std  = torch.tensor([0.229,0.224,0.225])[:,None,None]

    def __getitem__(self, idx):
        x_norm, y = super().__getitem__(idx)             # (3,H,W) normalized
        # de-normalize to [0,1] for SLIC
        x_vis = (x_norm*self.std + self.mean).clamp(0,1).permute(1,2,0).cpu().numpy()
        seg = slic(
            x_vis, n_segments=self.n_segments, compactness=self.compactness,
            start_label=0, channel_axis=-1
        ).astype(np.int32)                               # (H,W)
        return x_norm, torch.from_numpy(seg), y

def unfreeze_layer4(model):
    if hasattr(model, "backbone") and hasattr(model.backbone, "layer4"):
        for p in model.backbone.layer4.parameters():
            p.requires_grad = True

# ---------------- Simple CMF block on feature maps ----------------
class CMF(nn.Module):
    """
    Cross-Modal Fusion on feature maps:
    - project RGB feat map and FFT feat map to d_model
    - Q from RGB, K/V from FFT
    - residual back to RGB channel space (C_in)
    """
    def __init__(self, c_in=1024, d_model=256, heads=4, dropout=0.0):
        super().__init__()
        self.rgb_proj  = nn.Conv2d(c_in, d_model, 1, bias=False)
        self.freq_proj = nn.Conv2d(c_in, d_model, 1, bias=False)
        self.attn      = nn.MultiheadAttention(d_model, heads, batch_first=True, dropout=dropout)
        self.out       = nn.Conv2d(d_model, c_in, 1, bias=False)
        self.out_bn    = nn.BatchNorm2d(c_in)

    def forward(self, rgb_map, freq_map):
        B,C,H,W = rgb_map.shape
        q = self.rgb_proj(rgb_map).flatten(2).transpose(1,2)   # (B,HW,d)
        k = self.freq_proj(freq_map).flatten(2).transpose(1,2) # (B,HW,d)
        v = k
        fused,_ = self.attn(q,k,v)                             # (B,HW,d)
        fused = fused.transpose(1,2).reshape(B,-1,H,W)
        fused = self.out_bn(self.out(fused))
        return rgb_map + fused

# ---------------- Tiny transformer over tokens ----------------
class TransformerBlock(nn.Module):
    def __init__(self, dim, heads=4, mlp_ratio=4.0, dropout=0.0):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn  = nn.MultiheadAttention(dim, heads, batch_first=True, dropout=dropout)
        self.norm2 = nn.LayerNorm(dim)
        self.mlp   = nn.Sequential(
            nn.Linear(dim, int(dim*mlp_ratio)), nn.GELU(),
            nn.Linear(int(dim*mlp_ratio), dim)
        )
    def forward(self, x):
        x = x + self.attn(self.norm1(x), self.norm1(x), self.norm1(x))[0]
        x = x + self.mlp(self.norm2(x))
        return x

# ---------------- Superpixel token pooling ----------------
def superpixel_tokens(feat_map, seg_ids):
    """
    feat_map: (B,C,Hf,Wf)
    seg_ids : (B,H,W) integer ids; will be resized to (Hf,Wf) with nearest
    Returns:
      toks: (B,R,C) padded to max R in batch
      pad_mask: (B,R) True where padded
    """
    B,C,Hf,Wf = feat_map.shape
    seg = F.interpolate(seg_ids.unsqueeze(1).float(), size=(Hf,Wf), mode='nearest').squeeze(1).long()
    toks_list, counts = [], []
    for b in range(B):
        ids = torch.unique(seg[b])
        mapp = feat_map[b]          # (C,Hf,Wf)
        tokens_b = []
        for rid in ids:
            m = (seg[b]==rid).float()
            w = m / (m.sum()+1e-6)
            tok = (mapp * w).sum(dim=(1,2))   # (C,)
            tokens_b.append(tok)
        tokens_b = torch.stack(tokens_b,0)     # (Rb,C)
        toks_list.append(tokens_b)
        counts.append(tokens_b.size(0))
    maxR = max(counts)
    padded = []
    for t in toks_list:
        if t.size(0) < maxR:
            pad = torch.zeros(maxR - t.size(0), t.size(1), device=t.device, dtype=t.dtype)
            t = torch.cat([t, pad], dim=0)
        padded.append(t)
    toks = torch.stack(padded, 0)              # (B,R,C)
    pad_mask = torch.arange(maxR, device=toks.device)[None,:] >= torch.tensor(counts, device=toks.device)[:,None]
    return toks, pad_mask  # pad_mask=True means "ignore"

# ---------------- FFT magnitude map from input ----------------
@torch.no_grad()
def fft_mag_map(x_norm):
    """
    x_norm: (B,3,H,W) normalized; returns single-channel mag map ~ (B,1,H, W//2+1)
    """
    mean = torch.tensor([0.485,0.456,0.406], device=x_norm.device).view(1,3,1,1)
    std  = torch.tensor([0.229,0.224,0.225], device=x_norm.device).view(1,3,1,1)
    x = (x_norm*std + mean).clamp(0,1)
    y = 0.2989*x[:,0] + 0.5870*x[:,1] + 0.1140*x[:,2]   # (B,H,W)
    Y = torch.fft.rfft2(y, norm='ortho')
    mag = torch.log1p(torch.abs(Y)).unsqueeze(1)        # (B,1,H,W//2+1)
    return mag

# ---------------- Full Stage-2B model (SLIC tokens) ----------------
class P3Stage2B(nn.Module):
    def __init__(self, n_segments=50, imagenet_backbone=False,
                 enc_dim=1024, d_model=256, heads=4, enc_layers=2, num_classes=2):
        super().__init__()
        weights = ResNet50_Weights.IMAGENET1K_V1 if imagenet_backbone else None
        resnet = resnet50(weights=weights)
        self.backbone = nn.Sequential(
            resnet.conv1, resnet.bn1, resnet.relu, resnet.maxpool,
            resnet.layer1, resnet.layer2, resnet.layer3, resnet.layer4
        )
        self.neck = nn.Sequential(nn.Conv2d(2048, enc_dim, 1, bias=False),
                                  nn.BatchNorm2d(enc_dim), nn.ReLU(inplace=True))
        # project FFT mag (1ch) to enc_dim
        self.fft_proj = nn.Sequential(nn.Conv2d(1, enc_dim, 1, bias=False),
                                      nn.BatchNorm2d(enc_dim), nn.ReLU(inplace=True))
        self.cmf = CMF(c_in=enc_dim, d_model=d_model, heads=heads, dropout=0.0)

        self.encoder = nn.Sequential(*[TransformerBlock(enc_dim, heads=heads) for _ in range(enc_layers)])
        self.head = nn.Sequential(nn.LayerNorm(enc_dim),
                                  nn.Linear(enc_dim, num_classes))

    def forward(self, x, seg_ids):
        # backbone feats
        f = self.backbone(x)           # (B,2048,Hf,Wf)
        f = self.neck(f)               # (B,1024,Hf,Wf)

        # FFT branch on input, resize to feature map, project
        mag = fft_mag_map(x)                                         # (B,1,H,W//2+1)
        mag = F.interpolate(mag, size=f.shape[-2:], mode='bilinear', align_corners=False)
        f_freq = self.fft_proj(mag)                                  # (B,1024,Hf,Wf)

        # CMF residual fusion on maps
        f_fused = self.cmf(f, f_freq)                                # (B,1024,Hf,Wf)

        # Superpixel tokenization
        toks, pad_mask = superpixel_tokens(f_fused, seg_ids)         # (B,R,1024), (B,R)

        # Tiny transformer over tokens
        z = self.encoder(toks)                                       # (B,R,1024)

        # Global mean over valid tokens only
        valid = (~pad_mask).float()                                  # (B,R)
        denom = valid.sum(dim=1, keepdim=True).clamp_min(1.0)        # (B,1)
        pooled = (z * valid.unsqueeze(-1)).sum(dim=1) / denom        # (B,1024)

        logits = self.head(pooled)                                   # (B,2)
        return logits


In [3]:
# === Stage-2B: Imports (patched) ===
import os, sys, time, math, shutil, tempfile, traceback
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import DataLoader

# If your notebook defines them inline, keep that cell as-is and *do not* import here.

# Optional but recommended for slightly faster dataloaders on Kaggle
torch.backends.cudnn.benchmark = True


In [4]:
!ls -lh /kaggle/working | grep p3_stage2b_cmf.py || true


In [5]:
# === Stage-2B: Utils (patched) ===

def atomic_save(state, path: str):
    """Robust save that won't leave corrupt files on crash."""
    p = Path(path)
    p.parent.mkdir(parents=True, exist_ok=True)
    fd, tmp_path = tempfile.mkstemp(dir=str(p.parent))
    os.close(fd)
    torch.save(state, tmp_path)
    os.replace(tmp_path, str(p))

def zip_ckpts():
    """Zip all stage2b checkpoints to a single archive for easy download."""
    os.system("zip -j -q /kaggle/working/stage2b_ckpts.zip /kaggle/working/p3_stage2b_*.pth 2>/dev/null || true")

class AverageMeter:
    def __init__(self):
        self.reset()
    def reset(self):
        self.sum = 0.0
        self.count = 0
    @property
    def avg(self):
        return self.sum / max(1, self.count)
    def update(self, val, n=1):
        self.sum += float(val) * n
        self.count += n

@torch.no_grad()
def top1_accuracy(logits, y):
    return (logits.argmax(1) == y).float().mean().item()


In [6]:
# === Stage-2B: Freeze helpers (patched) ===

def apply_freezing(model, freeze_backbone: bool, freeze_encoder: bool):
    """
    freeze_backbone=True  -> freeze all resnet layers, we'll manually unfreeze layer4 below
    freeze_encoder=True   -> freeze transformer encoder (ViT-ish tiny transformer)
    """
    if freeze_backbone and hasattr(model, 'backbone'):
        for p in model.backbone.parameters():
            p.requires_grad = False
    if freeze_encoder and hasattr(model, 'encoder'):
        for p in model.encoder.parameters():
            p.requires_grad = False

def make_layer4_trainable(model):
    """
    Ensure last ResNet block is trainable (light unfreeze) — aligns with Stage-2B plan.
    """
    # If you already have unfreeze_layer4(model) imported from your module, call that instead.
    # This inline version assumes model.backbone.layer4 follows torchvision.resnet anatomy.
    if hasattr(model, 'backbone') and hasattr(model.backbone, 'layer4'):
        for p in model.backbone.layer4.parameters():
            p.requires_grad = True


In [7]:
# === Stage-2B Train/Eval (SLIC aware) ===
def evaluate(model, criterion, loader, device):
    model.eval()
    totL = totA = n = 0
    with torch.no_grad():
        for x, seg, y in loader:
            x, seg, y = x.to(device, non_blocking=True), seg.to(device), y.to(device)
            logits = model(x, seg)
            loss = criterion(logits, y)
            b = x.size(0)
            totL += loss.item() * b
            totA += (logits.argmax(1) == y).float().sum().item()
            n += b
    return totL/max(1,n), totA/max(1,n)

def train_one_epoch(model, criterion, optimizer, loader, device, max_grad_norm=None):
    model.train()
    totL = totA = n = 0
    for x, seg, y in loader:
        x, seg, y = x.to(device, non_blocking=True), seg.to(device), y.to(device)
        logits = model(x, seg)
        loss = criterion(logits, y)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        if max_grad_norm:
            nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
        optimizer.step()
        b = x.size(0)
        totL += loss.item() * b
        totA += (logits.argmax(1) == y).float().sum().item()
        n += b
    return totL/max(1,n), totA/max(1,n)


In [8]:
# === Stage-2B: Main Runner (patched) ===
import argparse
from datetime import datetime

def main():
    p = argparse.ArgumentParser()
    # Paths
    p.add_argument('--data_root', type=str, required=True)
    p.add_argument('--save', type=str, default='/kaggle/working/p3_stage2b_cmf.pth')
    p.add_argument('--ckpt', type=str, default=None, help='Optional init checkpoint (e.g., Stage-2)')
    # Train setup
    p.add_argument('--epochs', type=int, default=2)                 # keep light first
    p.add_argument('--batch_size', type=int, default=8)             # safer default; scale to 16/32 later
    p.add_argument('--lr', type=float, default=5e-4)
    p.add_argument('--weight_decay', type=float, default=1e-4)
    p.add_argument('--max_grad_norm', type=float, default=1.0)
    # Superpixels
    p.add_argument('--n_segments', type=int, default=25)            # lighter than 50; scale up later
    # Freezing: default = freeze (only layer4 + heads train)
    p.add_argument('--unfreeze_backbone', action='store_true')
    p.add_argument('--unfreeze_encoder', action='store_true')
    # If you still want to allow ImageNet backbone, keep the flag but default False
    p.add_argument('--imagenet_backbone', action='store_true',
                   help='Use only if you enabled Internet or have cached weights.')
    # Loader
    p.add_argument('--num_workers', type=int, default=4)
    p.add_argument('--pin_memory', action='store_true')

    args = p.parse_args()   # always read sys.argv

    # ---------------- Device ----------------
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"[{datetime.now()}] device: {device}")

      # ---------------- Dataset ----------------
    # Use the inline SLIC dataset defined earlier
    train_set = SuperpixelImageFolder(
        root=os.path.join(args.data_root, 'train'),
        n_segments=args.n_segments
    )
    val_set = SuperpixelImageFolder(
        root=os.path.join(args.data_root, 'valid'),
        n_segments=args.n_segments
    )

    train_loader = DataLoader(
        train_set, batch_size=args.batch_size, shuffle=True,
        num_workers=args.num_workers, pin_memory=args.pin_memory, drop_last=True
    )
    val_loader = DataLoader(
        val_set, batch_size=max(8, args.batch_size), shuffle=False,
        num_workers=args.num_workers, pin_memory=args.pin_memory
    )

    # ---------------- Model ----------------
    # Use the inline P3Stage2B defined earlier (no imports here)
    model = P3Stage2B(
        n_segments=args.n_segments,
        imagenet_backbone=args.imagenet_backbone  # stays False unless you pass the flag
    ).to(device)

    # Load init ckpt if provided
    if args.ckpt and os.path.isfile(args.ckpt):
        print(f"Loading init checkpoint: {args.ckpt}")
        state = torch.load(args.ckpt, map_location='cpu')
        # Accept partial matches
        missing, unexpected = model.load_state_dict(state if isinstance(state, dict) and 'state_dict' not in state else state, strict=False)
        print("load_state_dict (missing, unexpected):", len(missing), len(unexpected))

    # Apply freeze-by-default policy (unless user explicitly unfreezes)
    freeze_backbone = not args.unfreeze_backbone
    freeze_encoder  = not args.unfreeze_encoder
    apply_freezing(model, freeze_backbone, freeze_encoder)

    # Light-unfreeze last ResNet block always (Stage-2B plan)
    if 'unfreeze_layer4' in globals():
        unfreeze_layer4(model)
    else:
        make_layer4_trainable(model)


    # ---------------- Opt/crit ----------------
    criterion = nn.CrossEntropyLoss()
    # Only optimize trainable params
    trainable_params = [p for p in model.parameters() if p.requires_grad]
    optimizer = optim.AdamW(trainable_params, lr=args.lr, weight_decay=args.weight_decay)

    # ---------------- Train loop ----------------
    best_val_acc = -1.0
    best_path = args.save

    try:
        for epoch in range(1, args.epochs + 1):
            print(f"\n=== Epoch {epoch}/{args.epochs} ===")
            tr_loss, tr_acc = train_one_epoch(
                model, criterion, optimizer, train_loader, device, max_grad_norm=args.max_grad_norm
            )
            va_loss, va_acc = evaluate(model, criterion, val_loader, device)
            print(f"train: loss={tr_loss:.4f} acc={tr_acc:.4f} | val: loss={va_loss:.4f} acc={va_acc:.4f}")
           
            
            # ---- DeepfakeBench metrics on validation ----
            model.eval()
            y_true, y_prob = [], []
            with torch.no_grad():
                for x, seg, y in val_loader:
                    x, seg, y = x.to(device), seg.to(device), y.to(device)
                    probs = torch.softmax(model(x, seg), dim=1)[:, 1].cpu().numpy()
                    y_true.extend(y.cpu().numpy())
                    y_prob.extend(probs)
            
            # Calculate full DeepfakeBench metrics
            auc, f1, eer, acc = calc_metrics(np.array(y_true), np.array(y_prob))
            print(f"train: loss={tr_loss:.4f} acc={tr_acc:.4f} | "
                  f"val: AUROC={auc:.3f} | F1={f1:.3f} | EER={eer:.3f} | ACC={acc:.3f}")
            
            # Save checkpoint if AUROC improves
            if auc > best_val_acc:
                best_val_acc = auc
                atomic_save(model.state_dict(), best_path)
                print(f"[BEST] saved -> {best_path} (AUC={best_val_acc:.4f})")


            
            # Save "last" and zip after every epoch
            atomic_save(model.state_dict(), "/kaggle/working/p3_stage2b_last.pth")
            if va_acc > best_val_acc:
                best_val_acc = va_acc
                atomic_save(model.state_dict(), best_path)
                print(f"[BEST] saved -> {best_path} (val_acc={best_val_acc:.4f})")

            zip_ckpts()

    except KeyboardInterrupt:
        print("Interrupted — saving last checkpoint.")
        atomic_save(model.state_dict(), "/kaggle/working/p3_stage2b_last.pth")
        zip_ckpts()
        raise
    except Exception as e:
        print("Exception during training:\n", traceback.format_exc())
        print("Saving last checkpoint for debugging.")
        atomic_save(model.state_dict(), "/kaggle/working/p3_stage2b_last.pth")
        zip_ckpts()
        raise
    finally:
        # Always list outputs so you can download from Output tab after Save Version
        os.system("ls -lh /kaggle/working | egrep 'p3_stage2b_.*\\.pth|stage2b_ckpts\\.zip|train_log\\.txt' || true")

# if __name__ == "__main__":
#     main()


In [9]:
# === Stage-2B: Notebook Run (no here-doc, no runpy) ===
import sys
argv = [
  "run",
  "--data_root", "/kaggle/input/140k-real-and-fake-faces/real_vs_fake/real-vs-fake",
  "--epochs", "6",
  "--batch_size", "16",
  "--lr", "3e-4",
  "--n_segments", "50",
  "--pin_memory",
  "--imagenet_backbone",  
  "--save", "/kaggle/working/p3_stage2b_cmf.pth",
]


_argv_bak = sys.argv
try:
    sys.argv = argv
    main()     # <-- this calls the main() you defined in the runner cell
finally:
    sys.argv = _argv_bak


[2025-11-01 18:45:39.493133] device: cuda


Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth
100%|██████████| 97.8M/97.8M [00:00<00:00, 198MB/s]



=== Epoch 1/6 ===
train: loss=0.3672 acc=0.8418 | val: loss=0.2879 acc=0.8811
train: loss=0.3672 acc=0.8418 | val: AUROC=0.961 | F1=0.872 | EER=0.105 | ACC=0.881
[BEST] saved -> /kaggle/working/p3_stage2b_cmf.pth (AUC=0.9612)

=== Epoch 2/6 ===
train: loss=0.2708 acc=0.8908 | val: loss=0.2437 acc=0.9064
train: loss=0.2708 acc=0.8908 | val: AUROC=0.972 | F1=0.902 | EER=0.085 | ACC=0.906
[BEST] saved -> /kaggle/working/p3_stage2b_cmf.pth (AUC=0.9718)

=== Epoch 3/6 ===
train: loss=0.2210 acc=0.9128 | val: loss=0.2011 acc=0.9254
train: loss=0.2210 acc=0.9128 | val: AUROC=0.977 | F1=0.924 | EER=0.075 | ACC=0.925
[BEST] saved -> /kaggle/working/p3_stage2b_cmf.pth (AUC=0.9765)

=== Epoch 4/6 ===
train: loss=0.1903 acc=0.9261 | val: loss=0.1871 acc=0.9301
train: loss=0.1903 acc=0.9261 | val: AUROC=0.981 | F1=0.928 | EER=0.068 | ACC=0.930
[BEST] saved -> /kaggle/working/p3_stage2b_cmf.pth (AUC=0.9812)

=== Epoch 5/6 ===
train: loss=0.1640 acc=0.9372 | val: loss=0.2300 acc=0.9153
train: loss=0

In [10]:
!python -u run_stage2b.py \
  --data_root $DATA_ROOT \
  --epochs 0 \
  --n_segments 50 \
  --freeze_backbone --freeze_encoder \
  --ckpt /kaggle/working/p3_stage2b_cmf.pth


python3: can't open file '/kaggle/working/run_stage2b.py': [Errno 2] No such file or directory


In [11]:
!ls -lh /kaggle/working | egrep 'p3_stage2b_.*\.pth|stage2b_ckpts\.zip|train_log\.txt' || true


-rw------- 1 root root 199M Nov  2 00:07 p3_stage2b_cmf.pth
-rw------- 1 root root 199M Nov  2 00:07 p3_stage2b_last.pth
-rw-r--r-- 1 root root 364M Nov  2 00:08 stage2b_ckpts.zip
